# 🔬 Ray Serve + LLM 深度剖析

> **核心命题**：当一个模型大到任何单卡都装不下，或者你需要同时服务 100 个不同的模型——Ray 如何解决这个层次的问题？

前面的框架（vLLM, TensorRT-LLM, SGLang）都在优化**单节点单模型**的推理效率。
Ray Serve 解决的是**分布式、多模型、弹性伸缩**的问题——一个更「上层」的维度。

## 1. 为什么需要 Ray？

```
单节点单模型的局限:
  ┌──────────────────────┐
  │  GPU 0: Llama-70B    │  ← 模型太大？放不下
  │  GPU 1: Llama-70B    │  ← TP=2, 但还是这一台机器
  │  GPU 2: Llama-70B    │  ← 不能再加 GPU 了......
  │  GPU 3: Llama-70B    │
  └──────────────────────┘

Ray 视角:
  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
  │ Node 0       │  │ Node 1       │  │ Node 2       │
  │ GPU0: TP shard0│ │ GPU0: TP shard2│ │ GPU0: PP stage1│
  │ GPU1: TP shard1│ │ GPU1: TP shard3│ │ GPU1: PP stage2│
  │               │  │               │  │               │
  │ vLLM instance │  │ vLLM instance │  │ vLLM instance │
  │ (model A)     │  │ (model A)     │  │ (model B)     │
  └──────────────┘  └──────────────┘  └──────────────┘
       ↑                  ↑                  ↑
       └──────────────────┴──────────────────┘
                        │
              ┌─────────────────┐
              │  Ray Serve      │
              │  (路由 + 负载均衡)│
              └─────────────────┘
```

## 2. Ray Serve 架构

```
┌──────────────────────────────────────────────────────────────┐
│                     Ray Serve 架构                            │
├──────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │                    HTTP Proxy (Ray Actor)                │ │
│  │  • 接收外部请求                                          │ │
│  │  • 负载均衡到下游 Deployment Replicas                     │ │
│  │  • 请求队列 + 背压 (backpressure)                        │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │  Ray RPC (gRPC)                 │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │              Deployment Replicas (Ray Actors)           │ │
│  │                                                          │ │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐             │ │
│  │  │ Replica 0│  │ Replica 1│  │ Replica N│             │ │
│  │  │ ┌──────┐ │  │ ┌──────┐ │  │ ┌──────┐ │             │ │
│  │  │ │vLLM  │ │  │ │vLLM  │ │  │ │TRT-LLM│ │ ← 可以混用  │ │
│  │  │ │Worker│ │  │ │Worker│ │  │ │Worker│ │   不同引擎    │ │
│  │  │ └──────┘ │  │ └──────┘ │  │ └──────┘ │             │ │
│  │  └──────────┘  └──────────┘  └──────────┘             │ │
│  └─────────────────────────────────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │                   Ray Core                               │ │
│  │  • 分布式调度 (GCS - Global Control Store)               │ │
│  │  • 对象存储 (Plasma Store — 零拷贝共享)                  │ │
│  │  • 自动故障恢复 (Actor restart)                          │ │
│  │  • 自动扩缩容 (Autoscaler)                               │ │
│  └─────────────────────────────────────────────────────────┘ │
│                                                               │
└──────────────────────────────────────────────────────────────┘
```

## 3. 部署模式

### 3.1 Tensor Parallelism (TP)

```python
from ray import serve
from ray.serve.llm import LLMDeployment, LLMConfig

# 单个模型切分到多张 GPU
llm_config = LLMConfig(
    model_id="meta-llama/Llama-3.1-70B-Instruct",
    # TP=4: 模型权重、KV Cache 按 head 切分到 4 张 GPU
    tensor_parallelism=4,
    # 推理引擎
    engine="vllm",  # or "sglang", "tensorrt-llm"
    # 每个 GPU 上 1 个 replica
    replicas_per_gpu=1,
)

deployment = LLMDeployment.bind(llm_config)
serve.run(deployment, name="llama-70b")
```

### 3.2 Pipeline Parallelism (PP)

```python
# 模型按层切分: Layer 0-15 → GPU0, Layer 16-31 → GPU1
# 适合模型深度 >> 宽度的场景
llm_config = LLMConfig(
    model_id="meta-llama/Llama-3.1-70B-Instruct",
    pipeline_parallelism=2,  # 2 stages
    engine="vllm",
)
```

### 3.3 Data Parallelism (多 replica)

```python
# 同一个模型的多个独立副本——提升并发
llm_config = LLMConfig(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    # 4 个独立 replica，每个跑在 1 张 GPU 上
    replicas=4,
    # Ray Serve 自动做轮询负载均衡
)

# 等效架构:
#   R0 (GPU0): Llama-8B ← 请求 1,5,9,...
#   R1 (GPU1): Llama-8B ← 请求 2,6,10,...
#   R2 (GPU2): Llama-8B ← 请求 3,7,11,...
#   R3 (GPU3): Llama-8B ← 请求 4,8,12,...
# 吞吐: ~4× 单 replica
```

### 3.4 Model Ensemble (多模型组合)

```python
# Ray Serve 的真正威力: 组合多个模型
@serve.deployment
class Router:
    def __init__(self):
        self.classifier = serve.get_deployment("classifier")
        self.general = serve.get_deployment("general")
        self.coding = serve.get_deployment("coding")
    
    async def __call__(self, request: ChatRequest):
        # 第 1 步: 用小模型分类
        category = await self.classifier.classify.remote(request.prompt)
        
        # 第 2 步: 根据分类路由到不同的大模型
        if category == "coding":
            return await self.coding.chat.remote(request)
        else:
            return await self.general.chat.remote(request)

# 部署 3 个模型
classifier = LLMDeployment.bind(LLMConfig(model_id="small-classifier", replicas=2))
general = LLMDeployment.bind(LLMConfig(model_id="llama-70b", tensor_parallelism=4))
coding = LLMDeployment.bind(LLMConfig(model_id="deepseek-coder-33b", tensor_parallelism=2))
router = Router.bind()

# 这是前面所有框架都做不到的: 多个不同模型协同工作
```

## 4. 自动扩缩容 (Autoscaling)

```python
# Ray Serve 的自动扩缩容配置
llm_config = LLMConfig(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    # 扩容策略
    autoscaling_config={
        "min_replicas": 1,
        "max_replicas": 8,
        "target_num_ongoing_requests_per_replica": 4,  # 每个 replica 同时处理 4 个请求
        # 扩容: 如果每个 replica 的并发 > 4 → 增加 replica
        # 缩容: 如果每个 replica 的并发 < 1 → 减少 replica
        "upscale_delay_s": 30,   # 扩容冷却（避免抖动）
        "downscale_delay_s": 300, # 缩容冷却（保留多余的副本一段时间）
    }
)
```

## 5. Ray 的核心抽象：为什么它适合做 LLM 服务？

```
Ray Actor:
  概念: 一个有状态的分布式对象
  类比: 每个 Actor 是一个独立进程，有自己的内存空间
  
  @ray.remote(num_gpus=1)
  class LLMWorker:
      def __init__(self, model_id):
          self.model = load_model(model_id)  # 加载时占用 1 张 GPU
      
      def generate(self, prompt):
          return self.model.generate(prompt)  # 有状态，模型常驻 GPU

Ray Serve Deployment:
  概念: 一组同质的 Actor 副本 + 负载均衡
  类比: Kubernetes Deployment 的 Ray 版本

  @serve.deployment(num_replicas=4)
  class LLMService:
      ...

Ray Plasma Store:
  概念: 进程间零拷贝共享内存
  LLM 场景的关键用途: 共享权重（多 replica 共享同一份模型权重）
  
  # 正常情况下: 4 个 replica × 8GB 模型 = 32GB 显存
  # Plasma Store: 1 份权重 8GB + 4 份 KV Cache → ~20GB 显存
  # 节省 37% 显存
```

## 6. Ray Serve vs 直接裸跑 vLLM

| | Ray Serve + vLLM | 纯 vLLM |
|------|-----------------|---------|
| **单机单模型** | 额外开销，不建议 | ✅ 最佳选择 |
| **多模型** | ✅ 天然支持 | 需要自己写路由 |
| **多节点** | ✅ 自动发现 | 需要手动配置 |
| **弹性伸缩** | ✅ Autoscaler | 需要 K8s |
| **故障恢复** | ✅ Actor restart | 进程挂了就挂了 |
| **模型组合** | ✅ 最佳方案 | 不支持 |
| **部署复杂度** | 高 | 低 |

**选 Ray 的场景**：
- 需要同时服务多个不同模型（router + multiple backends）
- 需要弹性伸缩（流量波峰波谷明显）
- 多节点部署（一个模型太大，或需要冗余）
- 需要模型组合（router + classifier + generator 的 pipeline）

**不选 Ray 的场景**：
- 单机单模型、流量稳定 → 纯 vLLM 更快更简单
- 团队没有分布式系统经验 → Ray 学习曲线陡峭

## 7. 当前现实：Ray Serve + LLM 的成熟度

Ray Serve 的 LLM 支持（`ray.serve.llm`）还比较新（2024 年中推出），
最佳实践是：

```
当前推荐: vLLM 多节点部署 + 外部负载均衡 (如 nginx)
实验性:   Ray Serve + vLLM backend
未来:     Ray Serve LLMDeployment 成熟后统一
```

vLLM 本身已经支持 multi-node tensor parallelism（通过 Ray！），
所以实际上 vLLM 的多节点方案已经用了 Ray Core，只是没用 Ray Serve。

## 下一步

- → 返回 `../00-overview.ipynb` 查看完整的推理服务全景
- → `../server/03-vllm-deep-dive.ipynb`：理解 Ray 下层的推理引擎
- → `06-sglang-deep-dive.ipynb`：SGLang 也可以用 Ray 做分布式